# Study & credible set qualification

The goal of this notebook is to create a list of qualified studies and credible sets.


## Data Loading


In [7]:
# Download the release data from the Open Targets Platform 25.06 release
!rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.06/output/disease .
!rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.06/output/study .


Transfer starting: 2 files
disease/
disease/disease.parquet

sent 44 bytes  received 5142516 bytes  51425600000 bytes/sec
total size is 5348125  speedup is 1.04
Transfer starting: 3 files

sent 16 bytes  received 158 bytes  1740000 bytes/sec
total size is 93324727  speedup is 536345.92


In [ ]:
from enum import StrEnum

import pyspark.sql.functions as f
import pyspark.sql.types as t
from gentropy.common.session import Session
from gentropy.common.spark import string2camelcase
from pyspark.sql import Column, DataFrame


In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "10g"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/30 10:49:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/30 10:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 52580)
Traceback (most recent call last):
  File "/Users/ss60/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/ss60/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/Users/ss60/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/ss60/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Users/ss60/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/accumulators.py", lin

In [133]:
disease = session.spark.read.parquet("disease")
study = session.spark.read.parquet("study")
rescaled_betas = session.spark.read.parquet("lead_variant_effect")


This is the therapeutic area hierarchy:


## Methods


In [ ]:
class TherapeuticArea(StrEnum):
    """Therapeutic areas."""

    EFO_0001444 = "measurement"
    MONDO_0045024 = "cancer or benign tumor"
    OTAR_0000018 = "genetic familial or congenital disease"
    EFO_0005741 = "infectious disease"
    OTAR_0000009 = "injury poisoning or other complication"
    OTAR_0000014 = "pregnancy or perinatal disease"
    MONDO_0024458 = "disorder of visual system"
    EFO_0000319 = "cardiovascular disease"
    EFO_0009605 = "pancreas disease"
    EFO_0010282 = "gastrointestinal disease"
    OTAR_0000017 = "reproductive system or breast disease"
    EFO_0010285 = "integumentary system disease"
    EFO_0001379 = "endocrine system disease"
    OTAR_0000010 = "respiratory or thoracic disease"
    EFO_0009690 = "urinary system disease"
    OTAR_0000006 = "musculoskeletal or connective tissue disease"
    MONDO_0021205 = "disorder of ear"
    EFO_0000540 = "immune system disease"
    EFO_0005803 = "hematologic disease"
    EFO_0000618 = "nervous system disease"
    MONDO_0002025 = "psychiatric disorder"
    OTAR_0000020 = "nutritional or metabolic disease"
    EFO_0003765 = "sign or symptom"  # Not a therapeutic area - is descendant of phenotype
    # "EFO_0000651": "phenotype",
    # "GO_0008150":  "biological process",
    # "EFO_0002571":  "medical procedure",
    # "EFO_0005932": "animal disease",


def get_first_matching_therapeutic_area(
    ancestors: Column, therapeutic_area_hierarchy: type[TherapeuticArea] = TherapeuticArea
) -> Column:
    """Find the EFO entry ancestor that matches to the entry in therapeutic area hierarchy.

    Args:
        ancestors (Column): Column with EFO entry ancestries.
        therapeutic_area_hierarchy (type[TherapeuticArea]): Enum with therapeutic areas

    Returns:
        Column: with therapeutic area that match the entry with the lowest index in hierarchy.

    In case there are multiple matches, return the one with the lowest index in the enum.

    """
    tas = f.array(*[f.lit(ta.name).alias("name") for ta in therapeutic_area_hierarchy])
    ta_index = f.array(
        *[f.struct(f.lit(ta.name).alias("name"), f.lit(idx).alias("index")) for idx, ta in enumerate(TherapeuticArea)]
    )
    # Overlap tas and therapeutic_areas_array
    intersection = f.array_intersect(ancestors, tas)
    # Filter the index by the intersection
    intersection_index = (
        f.filter(ta_index, lambda x: f.array_contains(intersection, x.getField("name"))).getItem(0).getField("name")
    )
    return intersection_index


def get_efo_ta_index(disease: DataFrame) -> DataFrame:
    """Get the efo_ta lut."""
    efo_ta = (
        disease.select("id", "ancestors")
        .withColumn("primaryTherapeuticArea", get_first_matching_therapeutic_area(f.col("ancestors")))
        .withColumn(
            "primaryTherapeuticArea",
            f.when(f.col("primaryTherapeuticArea").isNull(), f.lit("other")).otherwise(f.col("primaryTherapeuticArea")),
        )
    )
    efo_ta_lookup = efo_ta.select("id", "primaryTherapeuticArea")
    return efo_ta_lookup


def assign_ta_to_study_index(study: DataFrame, efo_ta_lookup: DataFrame) -> DataFrame:
    """Assign primary therapeutic areas to the diseaseIds in study."""
    # Explode the study

    exploded_study = study.select(f.col("studyId"), f.explode("diseaseIds").alias("diseaseIdsExploded"))

    # Left join the efo_ta_lookup
    annotated_study = (
        exploded_study.join(efo_ta_lookup, how="left", on=efo_ta_lookup.id == exploded_study.diseaseIdsExploded)
        .withColumn("primaryTherapeuticAreaName", map_therapeutic_area_ids(f.col("primaryTherapeuticArea")))
        .drop("diseasIdsExploded", "primaryTherapeuticArea")
        .groupBy("studyId")
        .agg(f.collect_list("primaryTherapeuticAreaName").alias("mappedTherapeuticAreas"))
    )

    # Collect diseaseIds into array & filter out nulls
    return study.join(annotated_study, how="left", on="studyId")


def map_therapeutic_area_ids(ta: Column, therapeutic_area_hierarchy: type[TherapeuticArea] = TherapeuticArea) -> Column:
    """Map therapeutic area ids to names."""
    expr = f.when(f.lit(False), f.lit(None).cast(t.StringType()))

    for i in therapeutic_area_hierarchy:
        expr = expr.when(ta == f.lit(i.name), f.lit(i.value))

    return expr


def pivot_therapeutic_areas(study: DataFrame, ta_col: str = "mappedTherapeuticAreas") -> DataFrame:
    """Melt therapeutic areas column to separate columns.

    This operation converts long -> wide format.
    """
    exploded_study = (
        study.select(f.explode(ta_col).alias("ta"), f.col("studyId")).groupBy("studyId").pivot("ta").count()
    )
    columns_to_rename = [f.col(c).alias(string2camelcase(c)) for c in exploded_study.columns if c != "studyId"]
    columns_to_rename += [f.col("studyId")]

    ta_pivot = exploded_study.select(*columns_to_rename)
    return study.join(ta_pivot, on="studyId", how="left")


# Study-Index with therapeutic areas


In [114]:
efo_ta_lookup = get_efo_ta_index(disease)
gwas = study.filter(f.col("studyType") == "gwas")
gwas_ta = assign_ta_to_study_index(gwas, efo_ta_lookup)
gwas_ta_pivot = (
    pivot_therapeutic_areas(gwas_ta)
    .withColumn(
        "binaryLessCases",
        f.when(f.col("nCases") < f.col("nControls"), True).otherwise(False),
    )
    .withColumn("measurement", f.when(f.col("measurement") > 0, f.lit(True)).otherwise(False))
)
gwas_ta_pivot.show()


+--------------------+------+-----------+---------+--------------------+------------------------+---------------------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+--------------------+--------------------+-----------+----------------------+-------------------+---------------------+-------------+----------------------+----------------------+-----------------------+----------------------------------+------------------+-------------------+-----------------+----------------------------------+--------------------------+-----------+----------------------------------------+--------------------+-----------------------------+---------------+---------------------------+-------------------+-------------

In [115]:
(gwas_ta_pivot.filter(f.col("binaryLessCases")).groupBy("measurement").agg(f.count("studyId")).show())


+-----------+--------------+
|measurement|count(studyId)|
+-----------+--------------+
|       true|          3572|
|      false|         17788|
+-----------+--------------+



In [124]:
gwas_ta_pivot.write.parquet("gwas_therapeutic_areas", mode="overwrite")


# Qualifying studies


In [125]:
si_ta = session.spark.read.parquet("gwas_therapeutic_areas")


In [126]:
qualifying_studies = (
    si_ta.filter(f.col("binaryLessCases"))
    .filter(~f.col("measurement"))
    .filter(f.col("nSamples") > 10_000)
    .filter((f.col("nCases") / f.col("nSamples")) >= 0.005)
)
qualifying_studies.count()


9972

In [127]:
qualifying_studies.write.parquet("qualifying_studies", mode="overwrite")


# Qualifying measurements


In [128]:
# Removing protein measurements and microbiome descendants
# EFO_0007882 - microbiome
# EFO_0004747 - protein measurement

proteins_and_microbiome = (
    disease.select("id", "descendants")
    .filter(f.col("id").isin(["EFO_0007882", "EFO_0004747"]))
    .select(f.explode("descendants"))
)
proteins_and_microbiome_ids = [row["col"] for row in proteins_and_microbiome.collect()]
proteins_and_microbiome_ids.extend(["EFO_0007882", "EFO_0004747"])


In [129]:
len(proteins_and_microbiome_ids)


6293

In [130]:
qualifying_measurements = (
    si_ta.filter(f.col("measurement"))
    .filter(~f.col("binaryLessCases"))
    .filter(f.size(f.array_intersect(f.col("diseaseIds"), f.lit(proteins_and_microbiome_ids))) == 0)
)


In [131]:
qualifying_measurements.count()


61885

In [132]:
qualifying_measurements.write.parquet("qualifying_measurements", mode="overwrite")


# Qualified credible sets generation


In [137]:
qualifying_credible_sets = (
    rescaled_betas.join(qualifying_studies.select("studyId"), "studyId", "semi")
    .filter(f.col("majorLdPopulationMaf.value") > 0)
    .filter(f.abs("rescaledStatistics.estimatedBeta") <= 3)
)


In [138]:
qualifying_credible_sets.count()


72170

In [ ]:
qualifying_credible_sets.write.parquet("qualifying_credible_sets", mode="overwrite")


25/07/31 02:16:23 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 929500 ms exceeds timeout 120000 ms
25/07/31 02:16:23 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/31 02:31:33 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$